<a href="https://colab.research.google.com/github/SithmiDesilva/Hallucination-Aware-AI-Framework-for-Trustworthy-Supply-Chain-Decision-Intelligence.-/blob/News_data_scraping/News_Data_Extraction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Searching for Supply Chain News using Exa API

First, we need to install the official `exa_py` SDK.

In [1]:
!pip install exa_py

In [2]:
import os, json, time, re, datetime as dt
from urllib.parse import urlparse
import pandas as pd
from exa_py import Exa

# --- API key: Colab Secrets first, then environment variable ---
try:
    from google.colab import userdata
    exa_api_key = userdata.get('EXA_API_KEY')
except Exception:
    exa_api_key = os.environ.get('EXA_API_KEY', 'YOUR_EXA_API_KEY')

exa = Exa(api_key=exa_api_key)

# --- Save to Google Drive so files survive the Colab session ---
USE_DRIVE = True
if USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        OUT_DIR = '/content/drive/MyDrive/IRP/news_data'
    except Exception:
        OUT_DIR = 'news_data'
else:
    OUT_DIR = 'news_data'
os.makedirs(OUT_DIR, exist_ok=True)

# Every run gets an ID so results are reproducible / comparable later
RUN_ID = dt.datetime.now(dt.timezone.utc).strftime('%Y%m%dT%H%M%SZ')
RAW_LOG = os.path.join(OUT_DIR, f'raw_exa_log_{RUN_ID}.jsonl')
SEARCH_TYPE = 'neural'   # if Exa rejects this, use 'auto'
print('Run ID:', RUN_ID, '| output folder:', OUT_DIR)

Mounted at /content/drive
Run ID: 20261125T132210Z | output folder: /content/drive/MyDrive/IRP/news_data


In [3]:
# M5: 2011-01-29 to 2016-06-19 (CA, TX, WI)
# DataCo: orders ~Jan 2015 to Jan 2018 (global markets) - check against your copy:
#   pd.to_datetime(dataco['order date (DateOrders)']).agg(['min', 'max'])
DATASET_WINDOWS = {
    'M5':     ('2011-01-29', '2016-06-19'),
    'DataCo': ('2015-01-01', '2018-01-31'),
}
PUBLISH_BUFFER_DAYS = 30   # catch reporting published shortly after the window ends

def iso(d, end=False):
    return f'{d}T23:59:59.000Z' if end else f'{d}T00:00:00.000Z'

def yearly_windows(start, end):
    """Split a date range into calendar-year windows (one Exa search per year)."""
    s, e = pd.Timestamp(start), pd.Timestamp(end)
    out = []
    for y in range(s.year, e.year + 1):
        ws = max(s, pd.Timestamp(f'{y}-01-01'))
        we = min(e, pd.Timestamp(f'{y}-12-31'))
        out.append((ws.strftime('%Y-%m-%d'), we.strftime('%Y-%m-%d')))
    return out

def norm_url(u):
    """Normalise a URL so search hits and fetched contents match (http/https, www, trailing slash)."""
    if not isinstance(u, str):
        return None
    p = urlparse(u.strip())
    host = p.netloc.lower().removeprefix('www.')
    path = p.path.rstrip('/')
    return f'{host}{path}' + (f'?{p.query}' if p.query else '')

print(yearly_windows(*DATASET_WINDOWS['M5']))
print(yearly_windows(*DATASET_WINDOWS['DataCo']))

[('2011-01-29', '2011-12-31'), ('2012-01-01', '2012-12-31'), ('2013-01-01', '2013-12-31'), ('2014-01-01', '2014-12-31'), ('2015-01-01', '2015-12-31'), ('2016-01-01', '2016-06-19')]
[('2015-01-01', '2015-12-31'), ('2016-01-01', '2016-12-31'), ('2017-01-01', '2017-12-31'), ('2018-01-01', '2018-01-31')]
